# Crawling connected chats


In [1]:

import os
import asyncio
import json
import re
from collections import deque
from typing import Dict, Iterable, List, Set, Tuple

from telethon import TelegramClient
from telethon.errors import FloodWaitError
from telethon.tl.functions.channels import GetFullChannelRequest
from telethon.tl.functions.messages import GetFullChatRequest
from telethon.tl.types import Chat, Channel, Message, MessageEntityMention, MessageEntityMentionName, MessageEntityTextUrl


# --- config helpers ---------------------------------------------------------

def load_env_from_file(path: str = ".env") -> None:
    """Load KEY=VALUE lines into env if not already set."""
    if not os.path.exists(path):
        return
    with open(path, "r", encoding="utf-8") as fh:
        for line in fh:
            stripped = line.strip()
            if not stripped or stripped.startswith("#") or "=" not in stripped:
                continue
            key, _, value = stripped.partition("=")
            if key and value and key not in os.environ:
                os.environ[key] = value


def load_config() -> Dict:
    load_env_from_file()
    config = {
        "api_id": os.getenv("TG_API_ID"),
        "api_hash": os.getenv("TG_API_HASH"),
        "session_name": os.getenv("TG_SESSION_NAME", "session_name"),
        "start_seeds": [s.strip() for s in os.getenv("TG_START_SEEDS", "jungenationalisten").split(",") if s.strip()],
        "max_depth": int(os.getenv("TG_MAX_DEPTH", "2")),
        "max_per_chat": os.getenv("TG_MAX_PER_CHAT", "-1"),
        "output_path": os.getenv("TG_OUTPUT_PATH", "data/graph.json"),
        "handle_delay": float(os.getenv("TG_HANDLE_DELAY", "2")),
        "max_wait": int(os.getenv("TG_MAX_WAIT", "600")),
    }
    try:
        max_per_chat_val = int(config["max_per_chat"])
    except Exception:
        max_per_chat_val = -1
    config["max_per_chat"] = None if max_per_chat_val <= 0 else max_per_chat_val
    if not config["api_id"] or not config["api_hash"]:
        raise RuntimeError("Missing TG_API_ID/TG_API_HASH")
    return config


# --- parsing helpers -------------------------------------------------------

def _handle_from_url(url: str) -> str:
    match = re.search(r"(?:https?://)?t\.me/(?:c/\d+/)?([A-Za-z0-9_]+)", url or "")
    return match.group(1) if match else ""


def _parse_tme_message_link(url: str) -> Tuple[str, str]:
    match = re.search(r"(?:https?://)?t\.me/([A-Za-z0-9_]+)/([0-9]+)", url or "")
    return (match.group(1), match.group(2)) if match else ("", "")


def _collect_reactions(message: Message) -> Tuple[int, Dict[str, int]]:
    counts: Dict[str, int] = {}
    if message.reactions and getattr(message.reactions, "results", None):
        for reaction in message.reactions.results:
            emoji = getattr(reaction.reaction, "emoticon", str(reaction.reaction))
            counts[emoji] = counts.get(emoji, 0) + reaction.count
    return sum(counts.values()), counts


def _extract_mentions_and_links(message: Message) -> Tuple[Set[str], List[Tuple[str, str, str]]]:
    chats: Set[str] = set()
    links: List[Tuple[str, str, str]] = []
    if not message:
        return chats, links

    if message.entities:
        text = message.message or ""
        for entity in message.entities:
            if isinstance(entity, MessageEntityMention):
                handle = text[entity.offset : entity.offset + entity.length].lstrip("@")
                if handle:
                    chats.add(handle)
            elif isinstance(entity, MessageEntityMentionName):
                continue
            elif isinstance(entity, MessageEntityTextUrl):
                handle = _handle_from_url(entity.url)
                if handle:
                    chats.add(handle)
                msg_handle, msg_id = _parse_tme_message_link(entity.url)
                if msg_handle and msg_id:
                    links.append((msg_handle, msg_id, "link"))

    if message.forward and message.forward.chat:
        username = getattr(message.forward.chat, "username", None)
        if username:
            chats.add(username)
            fwd_msg_id = (
                getattr(message.forward, "channel_post", None)
                or getattr(message.forward, "saved_from_msg_id", None)
                or getattr(message.forward, "msg_id", None)
            )
            if fwd_msg_id:
                links.append((username, str(fwd_msg_id), "forward"))

    if message.message:
        for match in re.findall(r"(?:https?://)?t\.me/(?:c/\d+/)?([A-Za-z0-9_]+)", message.message):
            chats.add(match)
        for match in re.findall(r"(?:https?://)?t\.me/([A-Za-z0-9_]+)/([0-9]+)", message.message):
            chats.add(match[0])
            links.append((match[0], match[1], "link"))

    return chats, links


# --- Telegram helpers ------------------------------------------------------

def _get_title(entity, handle: str) -> str:
    return getattr(entity, "title", None) or getattr(entity, "first_name", "") or handle


async def _get_subscriber_count(client: TelegramClient, entity) -> int:
    try:
        if isinstance(entity, Channel):
            full = await client(GetFullChannelRequest(entity))
            return int(getattr(full.full_chat, "participants_count", 0) or 0)
        if isinstance(entity, Chat):
            full = await client(GetFullChatRequest(entity.id))
            return int(getattr(full.full_chat, "participants_count", 0) or 0)
    except Exception as exc:
        print(f"[warn] subscribers for {getattr(entity, 'username', entity)} failed: {exc}")
    return 0


async def _fetch_entity_with_backoff(client: TelegramClient, handle: str, max_wait: int):
    while True:
        try:
            return await client.get_entity(handle)
        except FloodWaitError as exc:
            wait = int(getattr(exc, "seconds", 0) or 0)
            if max_wait and wait > max_wait:
                print(f"[skip] {handle}: flood wait {wait}s exceeds max_wait={max_wait}s; skipping")
                return None
            print(f"[wait] {handle}: flood wait {wait}s on get_entity; sleeping...")
            await asyncio.sleep(wait + 1)
        except Exception as exc:
            print(f"[skip] {handle}: {exc}")
            return None


# --- crawler ---------------------------------------------------------------

async def crawl_graph(client: TelegramClient, seeds: Iterable[str], max_depth: int, max_per_chat, handle_delay: float, max_wait: int) -> Dict[str, List[Dict[str, str]]]:
    visited: Set[str] = set()
    queue: deque[Tuple[str, int]] = deque((seed, 0) for seed in seeds)
    nodes: Dict[str, Dict[str, str]] = {}
    edges: List[Dict[str, str]] = []
    messages: List[Dict[str, str]] = []
    message_edges: List[Dict[str, str]] = []

    while queue:
        handle, depth = queue.popleft()
        if handle in visited or depth > max_depth:
            continue

        if handle_delay > 0:
            await asyncio.sleep(handle_delay)

        entity = await _fetch_entity_with_backoff(client, handle, max_wait)
        if not entity:
            visited.add(handle)
            continue

        title = _get_title(entity, handle)
        subscribers = await _get_subscriber_count(client, entity)
        nodes[handle] = {"id": handle, "title": title, "subscribers": subscribers}
        visited.add(handle)
        print(f"[scrape] {handle} depth={depth}")

        while True:
            try:
                fetch_limit = None if max_per_chat is None else max_per_chat
                async for message in client.iter_messages(entity, limit=fetch_limit):
                    msg_key = f"{handle}:{message.id}"
                    reaction_count, reaction_details = _collect_reactions(message)
                    messages.append({
                        "id": msg_key,
                        "chat": handle,
                        "message_id": str(message.id),
                        "date": message.date.isoformat() if message.date else "",
                        "text": message.message or "",
                        "views": int(message.views or 0),
                        "sender_id": message.sender_id,
                        "reaction_count": reaction_count,
                        "reaction_breakdown": reaction_details,
                        "url": f"https://t.me/{handle}/{message.id}",
                    })

                    chat_mentions, msg_links = _extract_mentions_and_links(message)
                    for target in chat_mentions:
                        edges.append({"from": handle, "to": target, "message_id": str(message.id)})
                        if target not in visited and depth < max_depth:
                            queue.append((target, depth + 1))

                    for target_chat, target_msg_id, reason in msg_links:
                        message_edges.append({"from": msg_key, "to": f"{target_chat}:{target_msg_id}", "type": reason})

                    if getattr(message, "reply_to_msg_id", None):
                        message_edges.append({"from": msg_key, "to": f"{handle}:{message.reply_to_msg_id}", "type": "reply"})
                break
            except FloodWaitError as exc:
                wait = int(getattr(exc, "seconds", 0) or 0)
                if max_wait and wait > max_wait:
                    print(f"[skip] {handle}: flood wait {wait}s on iter_messages exceeds max_wait={max_wait}s; skipping chat")
                    break
                print(f"[wait] {handle}: flood wait {wait}s on iter_messages; sleeping and resuming...")
                await asyncio.sleep(wait + 1)
                continue
            except Exception as exc:
                print(f"[warn] iter_messages for {handle} failed: {exc}")
                break

    return {"nodes": list(nodes.values()), "edges": edges, "messages": messages, "message_edges": message_edges}


async def run_crawl(config):
    client = TelegramClient(config["session_name"], int(config["api_id"]), config["api_hash"])
    await client.start()
    print("Client connected")
    graph = await crawl_graph(client, config["start_seeds"], config["max_depth"], config["max_per_chat"], config["handle_delay"], config["max_wait"])
    os.makedirs(os.path.dirname(config["output_path"]) or ".", exist_ok=True)
    with open(config["output_path"], "w", encoding="utf-8") as fp:
        json.dump(graph, fp, ensure_ascii=False, indent=2)
    print(f"Wrote graph: {len(graph['nodes'])} nodes, {len(graph['edges'])} chat-edges, {len(graph['messages'])} messages, {len(graph['message_edges'])} message-edges -> {config['output_path']}")
    await client.disconnect()


config = load_config()
config


{'api_id': '27406534',
 'api_hash': '06dfedef1293c5a357b1a5a9cf29de49',
 'session_name': 'session_name',
 'start_seeds': ['jungenationalisten'],
 'max_depth': 2,
 'max_per_chat': 5000,
 'output_path': 'data/graph.json',
 'handle_delay': 2.0,
 'max_wait': 600}

In [2]:

import asyncio
try:
    asyncio.get_running_loop()
except RuntimeError:
    asyncio.run(run_crawl(config))
else:
    import nest_asyncio
    nest_asyncio.apply()
    await run_crawl(config)


Client connected
[scrape] jungenationalisten depth=0
[skip] JN_Th: flood wait 5324s exceeds max_wait=600s; skipping
[skip] heimat2023: flood wait 5322s exceeds max_wait=600s; skipping
[skip] schmidtkeswelt: flood wait 5320s exceeds max_wait=600s; skipping
[skip] dortmundsrechte: flood wait 5318s exceeds max_wait=600s; skipping
[skip] eimatDo!
: Cannot find any entity corresponding to "eimatDo!
"
[skip] mundsrechte: flood wait 5314s exceeds max_wait=600s; skipping
[skip] JN_Sachsen_Anhalt: flood wait 5312s exceeds max_wait=600s; skipping
[skip] JN_voran: flood wait 5310s exceeds max_wait=600s; skipping
[skip] HeimatSachsenAnhalt: flood wait 5308s exceeds max_wait=600s; skipping
[skip] JungeNationalisten: flood wait 5306s exceeds max_wait=600s; skipping
[skip] KDN2013: flood wait 5304s exceeds max_wait=600s; skipping
[skip] fsnrev: flood wait 5302s exceeds max_wait=600s; skipping
[skip] pcrecords: flood wait 5300s exceeds max_wait=600s; skipping
[skip] theresia_brand: flood wait 5298s ex

CancelledError: 